# 00 — Setup: Unity Catalog + Secrets + LLM endpoint

First notebook in the **Databricks + Bigdata.com MCP** demo. It provisions the
governance objects every later notebook depends on:

1. A **Unity Catalog** catalog + schema to hold the demo's tables, functions,
   vector index, and agent model.
2. A **Databricks secret scope** holding your Bigdata.com and OpenAI API keys, so keys
   are never written into notebook cells or the model artifact.
3. An **AI Gateway external-model endpoint** (`openai-chat`) that serves OpenAI `gpt-5.4`
   as the agent's LLM — the key stays in the secret; see Section 3 below for the full
   write-up.

### Requirements
- A Databricks workspace with **Unity Catalog** and **Serverless** enabled.
- Permission to create a catalog and a Model Serving endpoint.
- A **Bigdata.com API key** — [platform.bigdata.com/api-keys](https://platform.bigdata.com/api-keys).
- An **OpenAI API key** — [platform.openai.com/api-keys](https://platform.openai.com/api-keys).

> **Run order:** `00` → `01` → `02` → `03` → `04` → `05` → (`06` to clean up).

## Configuration

These names are the defaults used by every notebook in this demo. Change them here
and keep them consistent across the other notebooks (each notebook has the same
config block at the top).

In [ ]:
CATALOG = "bigdata_demo"
SCHEMA = "financial_intelligence"

# Secret scope + keys (one scope holds both API keys)
SECRET_SCOPE = "bigdata"
SECRET_KEY = "api_key"                 # Bigdata.com API key
OPENAI_SECRET_KEY = "openai_api_key"   # OpenAI API key (drives the agent's LLM)

# OpenAI LLM served via an AI Gateway external-model endpoint (created in Section 3 below).
# The agent reaches it as ChatDatabricks(endpoint=LLM_ENDPOINT_NAME).
LLM_ENDPOINT_NAME = "openai-chat"
OPENAI_MODEL = "gpt-5.4"               # use the exact model id your OpenAI account exposes

print(f"Catalog:      {CATALOG}")
print(f"Schema:       {CATALOG}.{SCHEMA}")
print(f"Bigdata key:  {{{{secrets/{SECRET_SCOPE}/{SECRET_KEY}}}}}")
print(f"OpenAI key:   {{{{secrets/{SECRET_SCOPE}/{OPENAI_SECRET_KEY}}}}}")
print(f"LLM endpoint: {LLM_ENDPOINT_NAME}  (model {OPENAI_MODEL})")

## 1. Create the catalog and schema

Unity Catalog is the governance layer for the whole demo — tables, functions, the
vector index, and the deployed agent model all live here and inherit its access
controls and lineage.

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

## 2. Store the Bigdata.com API key as a Databricks secret

Secrets keep the API key out of notebook source and out of the logged model. The
secret scope is created with the **Databricks CLI** (secret scopes cannot be
created from a notebook). Run these two commands in a terminal where the
[Databricks CLI](https://docs.databricks.com/dev-tools/cli/) is configured:

```bash
databricks secrets create-scope bigdata
databricks secrets put-secret bigdata api_key --string-value "bd-...your-key..."
```

The cell below verifies the secret is readable from this workspace.

In [ ]:
try:
    _key = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
    assert _key, "Secret is empty"
    print(f"OK — Bigdata.com API key found in scope '{SECRET_SCOPE}' "
          f"(length {len(_key)}, value redacted).")
except Exception as e:
    print("Secret not found yet. Create it with the Databricks CLI commands above, "
          "then re-run this cell.")
    print(f"Details: {e}")

## 3. Set up the agent's LLM — OpenAI via AI Gateway

The agent is driven by **OpenAI `gpt-5.4`**, served through a Databricks **AI Gateway
external-model endpoint** so the key stays in the secret scope and you get central
rate-limit control. (To use a Databricks-hosted model instead, skip this section and set
`LLM_ENDPOINT` to that model in `agent.py` / `05`.)

### 3a. Store the OpenAI API key

Add it to the same `bigdata` scope created above (run in a terminal with the Databricks CLI):

```bash
databricks secrets put-secret bigdata openai_api_key --string-value "sk-...your-openai-key..."
```

The next cell verifies it's readable.

In [ ]:
try:
    _oai = dbutils.secrets.get(scope=SECRET_SCOPE, key=OPENAI_SECRET_KEY)
    assert _oai, "Secret is empty"
    print(f"OK — OpenAI API key found in scope '{SECRET_SCOPE}' "
          f"(length {len(_oai)}, value redacted).")
except Exception as e:
    print("OpenAI secret not found yet. Run the CLI command above, then re-run this cell.")
    print(f"Details: {e}")

### 3b. Create the OpenAI external-model endpoint

This registers a Model Serving endpoint named **`openai-chat`** that proxies to OpenAI
`gpt-5.4`, reading the key from your secret via AI Gateway. Set the model to the exact id
your OpenAI account exposes — change `OPENAI_MODEL` in the config cell above if it differs
(e.g. `gpt-4.1`, `gpt-4o`). The cell is idempotent: re-run it to change the model.

In [ ]:
# Create (or update) the OpenAI external-model endpoint. Idempotent: re-running updates it.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ExternalModel,
    ExternalModelProvider,
    OpenAiConfig,
    ServedEntityInput,
)

w = WorkspaceClient()

served = ServedEntityInput(
    name="gpt-openai",
    external_model=ExternalModel(
        name=OPENAI_MODEL,                 # exact OpenAI model id (e.g. gpt-5.4)
        provider=ExternalModelProvider.OPENAI,
        task="llm/v1/chat",
        openai_config=OpenAiConfig(
            # Reference the secret — the key is never inlined here.
            openai_api_key=f"{{{{secrets/{SECRET_SCOPE}/{OPENAI_SECRET_KEY}}}}}",
        ),
    ),
)

existing = {e.name for e in w.serving_endpoints.list()}
if LLM_ENDPOINT_NAME in existing:
    w.serving_endpoints.update_config(name=LLM_ENDPOINT_NAME, served_entities=[served])
    print(f"Updated existing endpoint '{LLM_ENDPOINT_NAME}' (model {OPENAI_MODEL}).")
else:
    w.serving_endpoints.create(
        name=LLM_ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_entities=[served]),
    )
    print(f"Creating endpoint '{LLM_ENDPOINT_NAME}' (model {OPENAI_MODEL}) — give it a minute.")

In [ ]:
# Verify the endpoint with a quick chat call. First creation can take 1–2 minutes;
# if it errors as "not ready", wait a moment and re-run this cell.
from mlflow.deployments import get_deploy_client

try:
    resp = get_deploy_client("databricks").predict(
        endpoint=LLM_ENDPOINT_NAME,
        inputs={"messages": [{"role": "user", "content": "Reply with just: OK"}]},
    )
    print("OK — endpoint responded:", resp["choices"][0]["message"]["content"])
except Exception as e:
    print("Endpoint not ready yet (or errored) — wait ~1 minute and re-run this cell.")
    print(f"Details: {e}")

## Done

Catalog, schema, both secrets, and the `openai-chat` LLM endpoint are ready. Continue with
**`01_internal_data`** to load the internal portfolio tables and research documents.